1. Imports and configuration

In [ ]:
# Install legacy Keras matching TensorFlow 2.21 and TFMOT.
# Run this cell, restart the runtime, then run all cells from the top.
%pip install -q "tf-keras==2.21.0" "tensorflow-model-optimization==0.8.1"
print("Dependencies installed. Restart the runtime before running the notebook from the top.")

In [ ]:
import os
# Explicitly use tf_keras so model creation matches TFMOT legacy-Keras expectations.
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # set before importing TensorFlow

from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras as keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from datasets import load_dataset

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# SynCAN2025 is hosted on Hugging Face. Set SAMPLE_SIZE=None to use all rows.
DATASET_ID = "Thi-Thu-Huong/SynCAN2025"
SAMPLE_SIZE = None  # set None to use all 717k rows

print("TensorFlow:", tf.__version__)
print("Working directory:", Path.cwd())

2. Load SynCAN2025 from Hugging Face

In [ ]:
# Load SynCAN2025 directly from Hugging Face; no manual CSV download is needed.
# This requires an internet connection the first time and caches files locally.
ds = load_dataset(DATASET_ID)
print(ds)
print("Available splits:", list(ds.keys()))

split_name = "train" if "train" in ds else next(iter(ds.keys()))
hf_split = ds[split_name]
print("Columns:", hf_split.column_names)
print("Example:", hf_split[0])

# Convert to pandas. A reproducible shuffled sample keeps experimentation manageable.
df = hf_split.to_pandas()
if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
else:
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Rows loaded for this run:", len(df))
print("Label distribution:\n", df["Label"].value_counts(dropna=False))
display(df.head())

3. Feature engineering and label encoding

SynCAN payload strings are parsed into eight byte features; CAN ID and frame length provide the remaining two numeric features.

Label is the target class.

In [ ]:
# SynCAN2025 schema: Time, Protocol, Length, ID, Data, Label.
# Build a compact 10-feature vector: numeric CAN ID, DLC/Length, and 8 payload bytes.
required_cols = {"ID", "Length", "Data", "Label"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"SynCAN2025 schema mismatch. Missing columns: {sorted(missing_cols)}")

def parse_can_id(value):
    if pd.isna(value):
        return 0
    try:
        # SynCAN ID is usually an integer; support hex strings too.
        if isinstance(value, str) and value.lower().startswith("0x"):
            return int(value, 16)
        return int(value)
    except (TypeError, ValueError):
        try:
            return int(str(value), 16)
        except ValueError:
            return 0

def parse_payload(value, length):
    """Convert payload text into up to eight byte values, padded with zero."""
    out = [0] * 8
    if pd.isna(value):
        return out
    # Typical Data values are hex strings, sometimes space-separated.
    text = str(value).strip().replace("0x", "").replace("0X", "")
    tokens = text.split()
    try:
        if len(tokens) > 1:
            vals = [int(tok, 16) for tok in tokens[:8]]
        else:
            compact = text.replace(" ", "").replace("-", "").replace(":", "")
            if len(compact) % 2 == 1:
                compact = "0" + compact
            vals = [int(compact[i:i+2], 16) for i in range(0, min(len(compact), 16), 2)]
        vals = [v if 0 <= v <= 255 else 0 for v in vals]
        out[:min(8, len(vals))] = vals[:8]
    except ValueError:
        # Robust fallback for unexpected payload formatting.
        return out
    return out

feature_rows = []
for can_id, length, payload in zip(df["ID"], df["Length"], df["Data"]):
    bytes8 = parse_payload(payload, length)
    parsed_length = pd.to_numeric(length, errors="coerce")
    if pd.isna(parsed_length):
        parsed_length = 0
    feature_rows.append([parse_can_id(can_id), float(parsed_length), *bytes8])

feature_cols = ["CAN_ID_numeric", "DLC"] + [f"DATA[{i}]" for i in range(8)]
X = np.asarray(feature_rows, dtype=np.float32)
X[:, 1] = np.nan_to_num(X[:, 1], nan=0.0)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["Label"].astype(str)).astype(np.int64)
class_names = list(label_encoder.classes_)

print("Feature shape:", X.shape)
print("Classes:", class_names)
print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))

3.1. Exploratory data visuals

These plots describe the sampled dataset and engineered input features. They are generated from the data actually loaded by this run.

In [ ]:
# Exploratory dataset visuals — saved as PNG files as well as displayed inline.
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIG_DIR = Path("can_ids_results")
FIG_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 5))
label_counts = df["Label"].astype(str).value_counts()
label_counts.plot(kind="bar", ax=ax)
ax.set_title("Dataset class distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Frame count")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_class_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(11, 8))
corr_df = pd.DataFrame(X, columns=feature_cols).corr()
im = ax.imshow(corr_df.to_numpy(), aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols)), feature_cols, rotation=60, ha="right")
ax.set_yticks(range(len(feature_cols)), feature_cols)
ax.set_title("Feature correlation matrix")
fig.colorbar(im, ax=ax, label="Pearson correlation")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_feature_correlation.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    ax.hist(X[:, i], bins=35)
    ax.set_title(feature_cols[i])
    ax.set_ylabel("Count")
fig.suptitle("Feature distributions (raw engineered features)", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_feature_distributions.png", dpi=180, bbox_inches="tight")
plt.show()

print(f"Exploratory plots saved to: {FIG_DIR.resolve()}")

4. Stratified split and scaling

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
# Fit scaler on training data only to prevent test-set leakage.
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

class_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)
class_weight_dict = {int(k): float(v) for k, v in zip(np.unique(y_train), class_weights)}

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Class weights:", class_weight_dict)

5. Build and train the float32 baseline

In [ ]:
def build_model(input_dim, num_classes):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,), name="can_features"),
        keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(0.001)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(0.001)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation="relu", kernel_regularizer=keras.regularizers.l2(0.001)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(num_classes, activation="softmax", name="class_probabilities"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

baseline_model = build_model(X_train.shape[1], len(class_names))
baseline_model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-3),
]
baseline_history = baseline_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=80,
    batch_size=128,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

6. Evaluate the float32 baseline

In [ ]:
def evaluate_keras_model(model, X_eval, y_eval, title):
    probabilities = model.predict(X_eval, verbose=0)
    predictions = probabilities.argmax(axis=1)
    accuracy = accuracy_score(y_eval, predictions)
    print(f"{title} accuracy: {accuracy:.4%}")
    print(classification_report(
        y_eval, predictions, labels=np.arange(len(class_names)),
        target_names=class_names, zero_division  =0
    ))
    display(pd.DataFrame(
        confusion_matrix(y_eval, predictions, labels=np.arange(len(class_names))),
        index=[f"true_{c}" for c in class_names],
        columns=[f"pred_{c}" for c in class_names],
    ))
    return accuracy, predictions, probabilities

baseline_accuracy, baseline_predictions, baseline_probabilities = evaluate_keras_model(
    baseline_model, X_test, y_test, "Float32 baseline"
)

7. Prepare Quantization-Aware Training (QAT)

In [ ]:
# Dependencies are installed in the setup cell before TensorFlow is imported.
import tensorflow_model_optimization as tfmot
print("TensorFlow Model Optimization:", getattr(tfmot, "__version__", "version unknown"))


In [ ]:
try:
    import tensorflow_model_optimization as tfmot
except (ImportError, ModuleNotFoundError) as exc:
    raise ImportError(
        "Could not import tensorflow-model-optimization. Run the preceding install cell, "
        "restart the notebook kernel/runtime, then rerun cells from the top. "
        "If it still fails, check TensorFlow/Keras compatibility in this environment."
    ) from exc

print("TensorFlow Model Optimization:", getattr(tfmot, "__version__", "version unknown"))


8. Fine-tune a QAT model

In [ ]:
# QAT inserts fake-quantization operations during training. This requires the legacy
# keras API (tf_keras) used by tensorflow-model-optimization. If runtime still sees
# "to_quantize can only either be a keras Sequential or Functional model", the
# kernel was likely started before TF_USE_LEGACY_KERAS was set: restart and run all.
if not isinstance(baseline_model, (keras.Sequential, keras.Model)):
    raise TypeError(f"Unexpected baseline model type: {type(baseline_model)}")
try:
    qat_model = tfmot.quantization.keras.quantize_model(baseline_model)
except ValueError as exc:
    if "to_quantize can only either be a keras Sequential or Functional model" in str(exc):
        raise RuntimeError(
            "QAT/Keras compatibility mismatch. Run the dependency setup cell, restart the kernel, "
            "then run every cell from the top. Do not import TensorFlow before setting "
            "TF_USE_LEGACY_KERAS=1."
        ) from exc
    raise
qat_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
qat_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
]
qat_history = qat_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=25, #since patience is set, epochs might end before the limit set here
    batch_size=128,
    class_weight=class_weight_dict,
    callbacks=qat_callbacks,
    verbose=1,
)

9. Evaluate QAT model before TFLite conversion

In [ ]:
qat_accuracy, qat_predictions, qat_probabilities = evaluate_keras_model(
    qat_model, X_test, y_test, "QAT Keras model"
)

10. Convert to full INT8 TFLite

In [ ]:
# Representative samples calibrate activation ranges. Inputs to the exported model
# are quantized INT8; output is also requested as INT8.
def representative_dataset():
    n = min(1000, len(X_train))
    for sample in X_train[:n]:
        yield [sample.reshape(1, -1).astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

int8_tflite_model = converter.convert()
INT8_MODEL_PATH = Path("ids_model_qat_int8.tflite")
INT8_MODEL_PATH.write_bytes(int8_tflite_model)
print(f"Saved {INT8_MODEL_PATH} ({len(int8_tflite_model):,} bytes / {len(int8_tflite_model)/1024:.2f} KiB)")

11. Verify INT8 operators and evaluate TFLite predictions

In [ ]:
interpreter = tf.lite.Interpreter(model_content=int8_tflite_model)
#interpreter might invoke the deprecation warning, till Tf20 this works fine
# otherwise you can alternatively migrate to LiteRT from ai_edge_litert
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]

print("Input:", input_detail["shape"], input_detail["dtype"],
      "scale/zero-point:", input_detail["quantization"])
print("Output:", output_detail["shape"], output_detail["dtype"],
      "scale/zero-point:", output_detail["quantization"])

# Assert that model I/O is actually INT8.
assert input_detail["dtype"] == np.int8, "TFLite input is not INT8."
assert output_detail["dtype"] == np.int8, "TFLite output is not INT8."

def predict_int8_tflite(interpreter, X_eval):
    in_d = interpreter.get_input_details()[0]
    out_d = interpreter.get_output_details()[0]
    scale, zero_point = in_d["quantization"]
    if scale == 0:
        raise ValueError("Invalid input quantization scale.")
    results = []
    for row in X_eval:
        quantized = np.round(row / scale + zero_point)
        quantized = np.clip(quantized, -128, 127).astype(np.int8).reshape(1, -1)
        interpreter.set_tensor(in_d["index"], quantized)
        interpreter.invoke()
        q_output = interpreter.get_tensor(out_d["index"]).astype(np.float32)
        out_scale, out_zero = out_d["quantization"]
        dequantized = (q_output - out_zero) * out_scale
        results.append(dequantized[0])
    return np.asarray(results)

int8_probabilities = predict_int8_tflite(interpreter, X_test)
int8_predictions = int8_probabilities.argmax(axis=1)
int8_accuracy = accuracy_score(y_test, int8_predictions)
print(f"Full INT8 TFLite accuracy: {int8_accuracy:.4%}")
print(classification_report(
    y_test, int8_predictions, labels=np.arange(len(class_names)),
    target_names=class_names, zero_division=0
))
display(pd.DataFrame(
    confusion_matrix(y_test, int8_predictions, labels=np.arange(len(class_names))),
    index=[f"true_{c}" for c in class_names],
    columns=[f"pred_{c}" for c in class_names],
))

12. Float32 vs QAT INT8 differentiator

In [ ]:
# Convert the baseline to float32 TFLite as a size/reference comparison.
float_converter = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
float_tflite_model = float_converter.convert()
Path("ids_model_float32.tflite").write_bytes(float_tflite_model)

float32_size = len(float_tflite_model)
int8_size = len(int8_tflite_model)
size_reduction = (1 - int8_size / float32_size) * 100 if float32_size else float("nan")
baseline_vs_int8_agreement = float(np.mean(baseline_predictions == int8_predictions))
qat_vs_int8_agreement = float(np.mean(qat_predictions == int8_predictions))

comparison = pd.DataFrame([
    {"Model": "Keras float32 baseline", "Test accuracy": baseline_accuracy,
     "Model size (KiB)": baseline_model.count_params() * 4 / 1024,
     "Prediction agreement vs INT8": baseline_vs_int8_agreement},
    {"Model": "Keras QAT (fake quant)", "Test accuracy": qat_accuracy,
     "Model size (KiB)": np.nan,
     "Prediction agreement vs INT8": qat_vs_int8_agreement},
    {"Model": "TFLite full INT8 QAT", "Test accuracy": int8_accuracy,
     "Model size (KiB)": int8_size / 1024,
     "Prediction agreement vs INT8": 1.0},
])
display(comparison)
print(f"Serialized TFLite float32 size: {float32_size/1024:.2f} KiB")
print(f"Serialized TFLite INT8 size: {int8_size/1024:.2f} KiB")
print(f"Serialized TFLite size reduction: {size_reduction:.2f}%")
print(f"Accuracy delta (INT8 - baseline): {(int8_accuracy-baseline_accuracy)*100:+.3f} percentage points")
print(f"INT8 prediction agreement with float32 baseline: {baseline_vs_int8_agreement:.2%}")

13. Save labels, scaler, and model metadata

In [ ]:
# Save preprocessing metadata so embedded/inference code can reproduce the same mapping.
metadata = {
    "feature_columns": feature_cols,
    "class_names": class_names,
    "label_to_index": {str(name): int(i) for i, name in enumerate(class_names)},
    "scaler_min_": scaler.min_.tolist(),
    "scaler_scale_": scaler.scale_.tolist(),
    "scaler_data_min_": scaler.data_min_.tolist(),
    "scaler_data_max_": scaler.data_max_.tolist(),
    "input_quantization_scale": float(input_detail["quantization"][0]),
    "input_quantization_zero_point": int(input_detail["quantization"][1]),
    "output_quantization_scale": float(output_detail["quantization"][0]),
    "output_quantization_zero_point": int(output_detail["quantization"][1]),
    "baseline_accuracy": float(baseline_accuracy),
    "qat_keras_accuracy": float(qat_accuracy),
    "int8_tflite_accuracy": float(int8_accuracy),
}
Path("ids_qat_int8_metadata.json").write_text(json.dumps(metadata, indent=2))
baseline_model.save("ids_model_float32.keras")
qat_model.save("ids_model_qat.keras")
print("Saved:")
print(" - ids_model_float32.tflite")
print(" - ids_model_qat_int8.tflite")
print(" - ids_model_float32.keras")
print(" - ids_model_qat.keras")
print(" - ids_qat_int8_metadata.json")

Results

In [ ]:
# CONSOLIDATED CAN IDS VISUALIZATION DASHBOARD
# Run after all training, evaluation, and INT8 conversion cells.

from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import label_binarize

required_vars = [
    "df", "X", "y", "class_names", "feature_cols",
    "X_train", "X_test", "y_train", "y_test",
    "baseline_history", "qat_history",
    "baseline_accuracy", "qat_accuracy", "int8_accuracy",
    "baseline_predictions", "qat_predictions", "int8_predictions",
    "baseline_probabilities", "qat_probabilities", "int8_probabilities",
    "float32_size", "int8_size",
    "baseline_vs_int8_agreement", "qat_vs_int8_agreement",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(
        "Run the notebook's data, training, evaluation, and comparison "
        f"cells first. Missing variables: {missing}"
    )

OUT = Path("can_ids_all_visuals")
OUT.mkdir(exist_ok=True)

classes = list(class_names)
n_classes = len(classes)
y_true = np.asarray(y_test).astype(int)

predictions = {
    "Float32 baseline": np.asarray(baseline_predictions).astype(int),
    "QAT model": np.asarray(qat_predictions).astype(int),
    "Full INT8 TFLite": np.asarray(int8_predictions).astype(int),
}
probabilities = {
    "Float32 baseline": np.asarray(baseline_probabilities),
    "QAT model": np.asarray(qat_probabilities),
    "Full INT8 TFLite": np.asarray(int8_probabilities),
}

# Keep metric records in one table for CSV export.
metric_records = []
for model_name, pred in predictions.items():
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, pred, labels=np.arange(n_classes), zero_division=0
    )
    for i, cls in enumerate(classes):
        metric_records.append({
            "Model": model_name,
            "Class": cls,
            "Precision": p[i],
            "Recall": r[i],
            "F1-score": f1[i],
        })

per_class_metrics = pd.DataFrame(metric_records)
per_class_metrics.to_csv(OUT / "per_class_metrics.csv", index=False)

# Helper: save and display every figure.
def finish_plot(filename):
    plt.tight_layout()
    plt.savefig(OUT / filename, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()

fig, ax = plt.subplots(figsize=(10, 5))
df["Label"].astype(str).value_counts().plot(kind="bar", ax=ax)
ax.set_title("CAN Dataset: Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Frame count")
ax.tick_params(axis="x", rotation=35)
finish_plot("01_class_distribution.png")

fig, ax = plt.subplots(figsize=(11, 8))
corr = pd.DataFrame(X, columns=feature_cols).corr()
im = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(feature_cols)), feature_cols, rotation=60, ha="right")
ax.set_yticks(range(len(feature_cols)), feature_cols)
ax.set_title("Correlation Between Engineered CAN Features")
fig.colorbar(im, ax=ax, label="Pearson correlation")
finish_plot("02_feature_correlation.png")

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    ax.hist(np.asarray(X)[:, i], bins=35)
    ax.set_title(feature_cols[i])
    ax.set_ylabel("Count")
fig.suptitle("Distributions of Engineered CAN Features")
finish_plot("03_feature_distributions.png")

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
curve_specs = [
    (axes[0, 0], baseline_history, "accuracy", "Baseline Accuracy"),
    (axes[0, 1], baseline_history, "loss", "Baseline Loss"),
    (axes[1, 0], qat_history, "accuracy", "QAT Accuracy"),
    (axes[1, 1], qat_history, "loss", "QAT Loss"),
]
for ax, history, metric, title in curve_specs:
    h = history.history
    if metric in h:
        ax.plot(h[metric], label="Training")
    if f"val_{metric}" in h:
        ax.plot(h[f"val_{metric}"], label="Validation")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric.capitalize())
    ax.grid(True, alpha=0.3)
    ax.legend()
finish_plot("04_training_curves.png")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(
        y_true, pred, labels=np.arange(n_classes), normalize="true"
    )
    im = ax.imshow(cm, vmin=0, vmax=1, aspect="auto")
    ax.set_title(name)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_xticks(range(n_classes), classes, rotation=70, ha="right")
    ax.set_yticks(range(n_classes), classes)
fig.colorbar(im, ax=axes.tolist(), label="Fraction of true class")
fig.suptitle("Normalized Confusion Matrices")
finish_plot("05_confusion_matrices.png")

for metric in ["Precision", "Recall", "F1-score"]:
    pivot = per_class_metrics.pivot(
        index="Class", columns="Model", values=metric
    )
    ax = pivot.plot(kind="bar", figsize=(12, 5))
    ax.set_title(f"Per-Class {metric}")
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.3)
    finish_plot(f"06_{metric.lower().replace('-', '_')}.png")

accuracy_values = {
    "Float32 baseline": float(baseline_accuracy),
    "QAT model": float(qat_accuracy),
    "Full INT8 TFLite": float(int8_accuracy),
}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(list(accuracy_values), list(accuracy_values.values()))
ax.set_title("Test Accuracy Comparison")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=15)
for bar, value in zip(bars, accuracy_values.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{value:.2%}", ha="center"
    )
finish_plot("07_accuracy_comparison.png")

fig, ax = plt.subplots(figsize=(8, 5))
sizes = [float32_size / 1024, int8_size / 1024]
bars = ax.bar(["Float32 TFLite", "INT8 TFLite"], sizes)
ax.set_title("Serialized TFLite Model Size")
ax.set_ylabel("KiB")
for bar, value in zip(bars, sizes):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f} KiB", ha="center", va="bottom"
    )
finish_plot("08_model_size.png")

agreement = {
    "Baseline vs INT8": float(baseline_vs_int8_agreement),
    "QAT vs INT8": float(qat_vs_int8_agreement),
}
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(list(agreement), list(agreement.values()))
ax.set_title("Prediction Agreement With INT8 Model")
ax.set_ylabel("Agreement")
ax.set_ylim(0, 1.05)
for bar, value in zip(bars, agreement.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{value:.2%}", ha="center"
    )
finish_plot("09_prediction_agreement.png")

# Macro-average one-vs-rest ROC; skips classes lacking positives/negatives.
y_bin = label_binarize(y_true, classes=np.arange(n_classes))
if n_classes == 2:
    y_bin = np.column_stack([1 - y_true, y_true])

fig, ax = plt.subplots(figsize=(8, 6))
roc_drawn = False

for name, probs in probabilities.items():
    if probs.ndim != 2 or probs.shape[1] != n_classes:
        continue

    valid_curves = []
    for j in range(n_classes):
        if len(np.unique(y_bin[:, j])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, j], probs[:, j])
        valid_curves.append((fpr, tpr))

    if valid_curves:
        grid = np.linspace(0, 1, 250)
        mean_tpr = np.mean([
            np.interp(grid, fpr, tpr) for fpr, tpr in valid_curves
        ], axis=0)
        macro_auc = auc(grid, mean_tpr)
        ax.plot(grid, mean_tpr, label=f"{name} (AUC={macro_auc:.3f})")
        roc_drawn = True

if roc_drawn:
    ax.plot([0, 1], [0, 1], "--", label="Chance")
    ax.set_title("Macro-Average One-vs-Rest ROC")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.legend()
    ax.grid(True, alpha=0.3)
    finish_plot("10_roc_curves.png")
else:
    plt.close(fig)
    print("ROC plot skipped: insufficient class variation in test labels.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, probs) in zip(axes, probabilities.items()):
    confidence = np.max(probs, axis=1)
    correct = predictions[name] == y_true
    ax.hist(confidence[correct], bins=25, alpha=0.7, label="Correct")
    ax.hist(confidence[~correct], bins=25, alpha=0.7, label="Incorrect")
    ax.set_title(name)
    ax.set_xlabel("Maximum predicted probability")
    ax.set_ylabel("Frame count")
    ax.legend()
finish_plot("11_prediction_confidence.png")


summary = pd.DataFrame([
    {
        "Model": "Float32 baseline",
        "Test accuracy": baseline_accuracy,
        "TFLite size (KiB)": float32_size / 1024,
        "Agreement with INT8": baseline_vs_int8_agreement,
    },
    {
        "Model": "QAT model",
        "Test accuracy": qat_accuracy,
        "TFLite size (KiB)": np.nan,
        "Agreement with INT8": qat_vs_int8_agreement,
    },
    {
        "Model": "Full INT8 TFLite",
        "Test accuracy": int8_accuracy,
        "TFLite size (KiB)": int8_size / 1024,
        "Agreement with INT8": 1.0,
    },
])
summary.to_csv(OUT / "model_comparison_summary.csv", index=False)

print("\n=== MODEL COMPARISON SUMMARY ===")
display(summary)
print(f"\nAll visuals and CSVs saved in: {OUT.resolve()}")

# Bundle all PNG and CSV outputs into one archive.
zip_path = Path("can_ids_all_visuals.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(OUT.iterdir()):
        if file.is_file():
            zf.write(file, arcname=f"{OUT.name}/{file.name}")

print(f"Consolidated results archive: {zip_path.resolve()}")

**Payload Test Inspector**

      beyond this, all cells are payload test for dataset validation and structural verification

      if satisifed with the dataset integrity, can safely ignore all cells beyond this, anyhow any of the cell beyond this does not edit, retrain or alter the existing model

In [ ]:
# Convert X from a NumPy array to a DataFrame
# Reuse feature_cols from feature-engineering cell
X_df = pd.DataFrame(X, columns=feature_cols)

cols_to_check = [
    "DATA[0]", "DATA[1]", "DATA[2]", "DATA[3]",
    "DATA[4]", "DATA[5]", "DATA[6]", "DATA[7]"
]

print("Feature diagnostics:")
for col in cols_to_check:
    if col not in X_df.columns:
        print(f"{col}: NOT FOUND")
        continue

    s = pd.to_numeric(X_df[col], errors="coerce")

    print(
        f"{col}: missing={s.isna().sum()}, "
        f"unique={s.nunique(dropna=True)}, "
        f"std={s.std():.6f}, "
        f"min={s.min()}, max={s.max()}"
    )

# Calculate feature correlations
corr = X_df.corr(numeric_only=True)

print("\nFeatures with undefined correlations:")
print(corr.columns[corr.isna().any()].tolist())

print("\nPayload-byte correlation matrix:")
display(corr.loc[cols_to_check, cols_to_check].round(3))

In [ ]:
# Inspect original payloads from the dataset
print("Original Data examples:")
display(df[["Data"]].head(20))

# Check whether payloads contain nonzero values in the first two bytes
def inspect_payload(value):
    if pd.isna(value):
        return None

    # Convert a hexadecimal payload string into bytes
    s = str(value).strip().replace(" ", "").replace("0x", "")

    try:
        if len(s) % 2 != 0:
            return None
        return [int(s[i:i+2], 16) for i in range(0, len(s), 2)]
    except ValueError:
        return None

payloads = df["Data"].apply(inspect_payload)

print("\nFirst two bytes in original payloads:")
for i in range(2):
    values = payloads.apply(
        lambda p: p[i] if p is not None and len(p) > i else np.nan
    )
    print(f"Byte {i}: unique={values.nunique()}, "
          f"missing={values.isna().sum()}, "
          f"sample={values.dropna().unique()[:10]}")


In [ ]:
# Count the number of hexadecimal bytes in each original payload
payload_lengths = (
    df["Data"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.split()
    .str.len()
)

print("Payload length distribution:")
print(payload_lengths.value_counts().sort_index())

print("\nMinimum payload length:", payload_lengths.min())
print("Maximum payload length:", payload_lengths.max())
